In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

In [2]:
df = pd.read_csv("../data/loan_approval_dataset.csv")
print("Dataset loaded successfully.")

Dataset loaded successfully.


In [3]:
df.head()

,loan_id,no_of_dependents,education,self_employed,income_annum,loan_amount,loan_term,cibil_score,residential_assets_value,commercial_assets_value,luxury_assets_value,bank_asset_value,loan_status
0,1,2,Graduate,No,9600000,29900000,12,778,2400000,17600000,22700000,8000000,Approved
1,2,0,Not Graduate,Yes,4100000,12200000,8,417,2700000,2200000,8800000,3300000,Rejected
2,3,3,Graduate,No,9100000,29700000,20,506,7100000,4500000,33300000,12800000,Rejected
3,4,3,Graduate,No,8200000,30700000,8,467,18200000,3300000,23300000,7900000,Rejected
4,5,5,Not Graduate,Yes,9800000,24200000,20,382,12400000,8200000,29400000,5000000,Rejected


In [4]:
df.columns = df.columns.str.strip()
print(df.columns.tolist())

['loan_id', 'no_of_dependents', 'education', 'self_employed', 'income_annum', 'loan_amount', 'loan_term', 'cibil_score', 'residential_assets_value', 'commercial_assets_value', 'luxury_assets_value', 'bank_asset_value', 'loan_status']


In [5]:
X = df.drop(columns=["loan_id", "loan_status"])
y = df["loan_status"]

print("X shape:", X.shape)
print("y shape:", y.shape)

X shape: (4269, 11)
y shape: (4269,)


In [6]:
print("Features:")
print(X.head())

print("\nTarget:")
print(y.head())

Features:
   no_of_dependents      education self_employed  income_annum  loan_amount  \
0                 2       Graduate            No       9600000     29900000   
1                 0   Not Graduate           Yes       4100000     12200000   
2                 3       Graduate            No       9100000     29700000   
3                 3       Graduate            No       8200000     30700000   
4                 5   Not Graduate           Yes       9800000     24200000   

   loan_term  cibil_score  residential_assets_value  commercial_assets_value  \
0         12          778                   2400000                 17600000   
1          8          417                   2700000                  2200000   
2         20          506                   7100000                  4500000   
3          8          467                  18200000                  3300000   
4         20          382                  12400000                  8200000   

   luxury_assets_value  bank_asset

In [7]:
categorical_cols = X.select_dtypes(include=["object"]).columns.tolist()
numeric_cols = X.select_dtypes(include=["int64", "float64"]).columns.tolist()

print("Categorical columns:", categorical_cols)
print("Numeric columns:", numeric_cols)

Categorical columns: ['education', 'self_employed']
Numeric columns: ['no_of_dependents', 'income_annum', 'loan_amount', 'loan_term', 'cibil_score', 'residential_assets_value', 'commercial_assets_value', 'luxury_assets_value', 'bank_asset_value']


/var/folders/93/089f0j5140590b91wt26qzwm0000gn/T/ipykernel_18449/77249225.py:1: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical_cols = X.select_dtypes(include=["object"]).columns.tolist()


In [8]:
for col in categorical_cols:
    print(f"{col}: {X[col].unique()}")

education: <StringArray>
[' Graduate', ' Not Graduate']
Length: 2, dtype: str
self_employed: <StringArray>
[' No', ' Yes']
Length: 2, dtype: str


In [9]:
for col in categorical_cols:
    X[col] = X[col].str.strip()

y = y.str.strip()

In [10]:
for col in categorical_cols:
    print(f"{col}: {X[col].unique()}")

print("loan_status:", y.unique())

education: <StringArray>
['Graduate', 'Not Graduate']
Length: 2, dtype: str
self_employed: <StringArray>
['No', 'Yes']
Length: 2, dtype: str
loan_status: <StringArray>
['Approved', 'Rejected']
Length: 2, dtype: str


In [11]:
X_encoded = X.copy()

feature_encoders = {}

for col in categorical_cols:
    le = LabelEncoder()
    X_encoded[col] = le.fit_transform(X_encoded[col])
    feature_encoders[col] = le

print(X_encoded.head())

   no_of_dependents  education  self_employed  income_annum  loan_amount  \
0                 2          0              0       9600000     29900000   
1                 0          1              1       4100000     12200000   
2                 3          0              0       9100000     29700000   
3                 3          0              0       8200000     30700000   
4                 5          1              1       9800000     24200000   

   loan_term  cibil_score  residential_assets_value  commercial_assets_value  \
0         12          778                   2400000                 17600000   
1          8          417                   2700000                  2200000   
2         20          506                   7100000                  4500000   
3          8          467                  18200000                  3300000   
4         20          382                  12400000                  8200000   

   luxury_assets_value  bank_asset_value  
0             22700

In [12]:
for col, le in feature_encoders.items():
    print(f"{col}:")
    for original, encoded in zip(le.classes_, le.transform(le.classes_)):
        print(f"  {original} -> {encoded}")

education:
  Graduate -> 0
  Not Graduate -> 1
self_employed:
  No -> 0
  Yes -> 1


In [13]:
target_encoder = LabelEncoder()
y_encoded = target_encoder.fit_transform(y)

print("Target classes:", target_encoder.classes_)
print("First 10 encoded target values:", y_encoded[:10])

Target classes: ['Approved' 'Rejected']
First 10 encoded target values: [0 1 1 1 1 1 0 1 0 1]


In [14]:
print("Target mapping:")
for original, encoded in zip(target_encoder.classes_, target_encoder.transform(target_encoder.classes_)):
    print(f"{original} -> {encoded}")

Target mapping:
Approved -> 0
Rejected -> 1


In [15]:
X_train, X_test, y_train, y_test = train_test_split(
    X_encoded,
    y_encoded,
    test_size=0.2,
    random_state=42,
    stratify=y_encoded
)

print("X_train shape:", X_train.shape)
print("X_test shape:", X_test.shape)
print("y_train shape:", y_train.shape)
print("y_test shape:", y_test.shape)

X_train shape: (3415, 11)
X_test shape: (854, 11)
y_train shape: (3415,)
y_test shape: (854,)


In [16]:
print("Full dataset class distribution:")
print(pd.Series(y_encoded).value_counts(normalize=True))

print("\nTraining set class distribution:")
print(pd.Series(y_train).value_counts(normalize=True))

print("\nTest set class distribution:")
print(pd.Series(y_test).value_counts(normalize=True))

Full dataset class distribution:
0    0.62216
1    0.37784
Name: proportion, dtype: float64

Training set class distribution:
0    0.622255
1    0.377745
Name: proportion, dtype: float64

Test set class distribution:
0    0.62178
1    0.37822
Name: proportion, dtype: float64


In [17]:
X_train.to_csv("../data/X_train.csv", index=False)
X_test.to_csv("../data/X_test.csv", index=False)

pd.Series(y_train, name="loan_status").to_csv("../data/y_train.csv", index=False)
pd.Series(y_test, name="loan_status").to_csv("../data/y_test.csv", index=False)

print("Train/test CSV files saved successfully.")

Train/test CSV files saved successfully.


## Step 3 observations

- Dropped `loan_id` because it is just an identifier.
- Set `loan_status` as the target column.
- Split data into:
  - `X` = input features
  - `y` = target labels
- Identified categorical columns and encoded them into numbers.
- Encoded the target labels into numeric form.
- Split the dataset into training and testing sets using an 80/20 split.
- Used stratification to preserve class balance across train and test sets.